# Part 2: Give it eyes

What happens: grab a frame, shrink it, JPEG it, base64 it, and put it in the chat
message next to your question. The model's vision encoder turns the picture into a
few hundred "image tokens". After that, it's just more prompt.

In [ ]:
# Run me first (Shift+Enter). Re-run me if you restart the kernel.
%load_ext autoreload
%autoreload 2

import config
from agent import nb, ears, mouth
from agent.llm import LocalLLM, system, user, assistant

llm = LocalLLM()
print("brain:", llm.model_name() if llm.health() else f"NOT RUNNING at {config.SERVER_URL} (run the Step 0 cells in 1_setup.ipynb)")

In [ ]:
from agent.eyes import Eyes, preprocess, save
import cv2, numpy as np

CAMERA = config.CAMERA        # 0 = webcam, or "samples/test_card.png", or a folder of photos
eyes = Eyes(CAMERA)

frame = eyes.frame()          # re-run this cell to grab a fresh frame
nb.show(frame)
print(frame.shape, "(height, width, colour channels)")

## Step by step: frame to prompt

Nothing hidden. Change `MAX_SIDE` and `JPEG_QUALITY` and watch the numbers.

In [ ]:
MAX_SIDE = 512
JPEG_QUALITY = 80

jpeg = preprocess(frame, max_side=MAX_SIDE, jpeg_quality=JPEG_QUALITY)
b64_len = len(jpeg) * 4 // 3

print(f"camera frame : {frame.shape[1]}x{frame.shape[0]} pixels, {frame.nbytes / 1e6:.1f} MB raw")
print(f"after shrink : longest side {MAX_SIDE}px")
print(f"as JPEG      : {len(jpeg) / 1024:.1f} KB")
print(f"as base64    : ~{b64_len / 1024:.1f} KB of text inside the JSON")
nb.show(nb.blow_up(jpeg), width=420)   # blown back up, so you can see what got lost

## Ask it something

Ask things a text-only model never could: *how many fingers am I holding up?*,
*read the text on my mug*, *is my plant thirsty?*

In [ ]:
QUESTION = "Describe what you see in one or two sentences. Be specific."
SYSTEM_PROMPT = "You are looking through a laptop webcam. Only describe what is actually visible."
GRAYSCALE = False
CROP = 1.0            # 1.0 = whole frame, 0.5 = middle half (free zoom)

frame = eyes.frame()
jpeg = preprocess(frame, max_side=MAX_SIDE, grayscale=GRAYSCALE, center_crop=CROP)
nb.show(nb.blow_up(jpeg), width=360)

r = llm.chat([system(SYSTEM_PROMPT), user(QUESTION, images=[jpeg])],
             on_token=nb.stream, temperature=0.2, max_tokens=120)
print("\n\n", r.stats())

## The sweep: speed vs detail

Same frame, six sizes. Watch **prompt tokens**. On some models it barely moves
(they resize everything to one fixed size inside). On others it grows with the
pixels. Which kind did you load?

In [ ]:
SWEEP_QUESTION = "What is the main object in this image? Five words max."

print(f"{'size':>6} {'KB':>6} {'prompt tok':>11} {'seconds':>8}   answer")
for side in (128, 224, 384, 512, 768, 1024):
    j = preprocess(frame, max_side=side)
    r = llm.chat([user(SWEEP_QUESTION, images=[j])], stream=False, max_tokens=30, temperature=0)
    print(f"{side:>6} {len(j) / 1024:>6.1f} {str(r.prompt_tokens or '?'):>11} {r.seconds:>8.2f}   {r.text.strip()[:60]}")

## The control panel

Sliders for everything above. **Snap** grabs a new frame, **Ask** sends it.
The picture shows exactly what the model gets.

In [ ]:
panel = nb.eyes_panel(llm, eyes, QUESTION, SYSTEM_PROMPT)

## Your turn: write your own preprocessing

`my_preprocess` gets the raw frame and returns a frame. Some ideas are below,
commented out. Turn one on, run, then run the ask cell under it.

In [ ]:
def my_preprocess(img):
    out = img.copy()

    # 1. Selfie mode: mirror it. Then ask "which hand am I raising?"
    # out = cv2.flip(out, 1)

    # 2. Dark room? Boost brightness and contrast.
    # out = cv2.convertScaleAbs(out, alpha=1.5, beta=20)

    # 3. A labelled 3x3 grid, so you can ask "which square is my phone in?"
    # h, w = out.shape[:2]
    # for i in (1, 2):
    #     cv2.line(out, (w * i // 3, 0), (w * i // 3, h), (0, 255, 255), 3)
    #     cv2.line(out, (0, h * i // 3), (w, h * i // 3), (0, 255, 255), 3)
    # for r_, row in enumerate("ABC"):
    #     for c_ in range(3):
    #         cv2.putText(out, f"{row}{c_ + 1}", (w * c_ // 3 + 15, h * r_ // 3 + 50),
    #                     cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 255), 3)

    return out

custom = my_preprocess(eyes.frame())
nb.show(custom)

In [ ]:
MY_QUESTION = "Describe what you see."
r = llm.chat([user(MY_QUESTION, images=[preprocess(custom)])], on_token=nb.stream, max_tokens=120)

## Try this

1. Hold up a book. In the panel, drag `max_side` down until it can't read the title.
2. Tick **grayscale**, ask "what colour is my shirt?". Does it admit it can't tell?
3. Crop to 0.4 and ask about something small in the middle of the frame.
4. Point it at a clock. Small models are famously bad at analogue clocks.

When you're done with this notebook, release the camera so the next one can use it:

In [ ]:
eyes.close()